# Submission 06: CatBoost Cross-Validation Winner

This submission uses the CatBoost configuration from Experiment 15, which was the strongest model in Experiment 21 after evaluating the model families using stratified 5-fold cross-validation.

Experiment 21 achieved a 5-fold mean accuracy of 0.8417 with a standard deviation of 0.0177. The model is retrained on the full training dataset before generating predictions for the Kaggle test set.

## 1. Setup

In [1]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from catboost import CatBoostClassifier


## 2. Load Data

In [2]:
train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')

X = train.drop(columns=['Survived'])
y = train['Survived']

X_test = test.copy()

print('Train shape:', train.shape)
print('Test shape:', test.shape)


Train shape: (891, 12)
Test shape: (418, 11)


## 3. Feature Engineering

In [3]:
def feature_engineering(df):
    df = df.copy()

    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

    df['AgeMissing'] = df['Age'].isna().astype(int)
    df['FareMissing'] = df['Fare'].isna().astype(int)
    df['EmbarkedMissing'] = df['Embarked'].isna().astype(int)

    df['HasCabin'] = df['Cabin'].notna().astype(int)
    df['CabinDeck'] = df['Cabin'].fillna('Unknown').astype(str).str[0]

    df['Title'] = df['Name'].str.extract(r',\s*([^.]*)\.')[0].str.strip()
    df['Title'] = df['Title'].replace({'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'})
    common_titles = ['Mr', 'Miss', 'Mrs', 'Master']
    df.loc[~df['Title'].isin(common_titles), 'Title'] = 'Rare'

    df['Surname'] = df['Name'].str.split(',').str[0].str.strip()

    df['TicketPrefix'] = (
        df['Ticket'].astype(str)
        .str.replace(r'\d', '', regex=True)
        .str.replace(r'[./ ]', '', regex=True)
        .replace('', 'NONE')
    )

    df['TicketGroupSize'] = df.groupby('Ticket')['Ticket'].transform('count')
    df['SurnameGroupSize'] = df.groupby('Surname')['Surname'].transform('count')

    df['FarePerPerson'] = df['Fare'] / df['TicketGroupSize'].replace(0, np.nan)
    df['FarePerPersonMissing'] = df['FarePerPerson'].isna().astype(int)

    df['SexPclass'] = df['Sex'].astype(str) + '_' + df['Pclass'].astype(str)

    df['FamilySizeBand'] = pd.cut(
        df['FamilySize'],
        bins=[0, 1, 4, 7, np.inf],
        labels=['Alone', 'Small', 'Medium', 'Large']
    ).astype(str)

    df['AgeBand'] = pd.cut(
        df['Age'],
        bins=[-np.inf, 12, 18, 30, 50, np.inf],
        labels=['Child', 'Teen', 'YoungAdult', 'Adult', 'Senior']
    ).astype(str)

    df['FareBand'] = pd.qcut(
        df['Fare'],
        q=5,
        labels=['VeryLow', 'Low', 'Medium', 'High', 'VeryHigh'],
        duplicates='drop'
    ).astype(str)

    df['FamilySex'] = df['Sex'].astype(str) + '_' + df['FamilySizeBand'].astype(str)
    df['PclassTitle'] = df['Pclass'].astype(str) + '_' + df['Title'].astype(str)
    df['PclassAgeBand'] = df['Pclass'].astype(str) + '_' + df['AgeBand'].astype(str)
    df['FamilyTicket'] = df['FamilySize'].astype(str) + '_' + df['TicketPrefix'].astype(str)

    df['NameLength'] = df['Name'].astype(str).str.len()
    df['NameWords'] = df['Name'].astype(str).str.split().str.len()
    df['TicketLength'] = df['Ticket'].astype(str).str.len()
    df['CabinCount'] = df['Cabin'].fillna('').astype(str).str.split().str.len()
    df['DeckKnown'] = df['Cabin'].notna().astype(int)

    df['LargeFamily'] = (df['FamilySize'] >= 5).astype(int)
    df['SmallFamily'] = ((df['FamilySize'] >= 2) & (df['FamilySize'] <= 4)).astype(int)
    df['FemaleChild'] = (
        (df['Sex'] == 'female') &
        (df['Age'].fillna(-1) <= 12)
    ).astype(int)

    df['FarePerAge'] = df['Fare'] / df['Age'].replace(0, np.nan)
    df['ClassFare'] = df['Pclass'] * df['Fare']
    df['SiblingChildRatio'] = df['SibSp'] / (df['Parch'] + 1)
    df['FamilyFare'] = df['Fare'] / df['FamilySize'].replace(0, np.nan)
    df['SexTitle'] = df['Sex'].astype(str) + '_' + df['Title'].astype(str)

    return df


X = feature_engineering(X)
X_test = feature_engineering(X_test)

drop_columns = ['PassengerId', 'Name', 'Ticket', 'Cabin', 'Surname']

X = X.drop(columns=drop_columns)
X_test = X_test.drop(columns=drop_columns)

print('Engineered train shape:', X.shape)
print('Engineered test shape:', X_test.shape)


Engineered train shape: (891, 41)
Engineered test shape: (418, 41)


## 4. Preprocessing

In [4]:
numeric_features = X.select_dtypes(include=['number']).columns.tolist()
categorical_features = X.select_dtypes(exclude=['number']).columns.tolist()

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_features),
    ('cat', categorical_pipeline, categorical_features)
])


## 5. CatBoost Model

In [5]:
model = CatBoostClassifier(
    iterations=600,
    depth=5,
    learning_rate=0.03,
    loss_function='Logloss',
    verbose=False,
    random_seed=42,
    l2_leaf_reg=5
)

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', model)
])


## 6. Train on Full Dataset and Predict

In [6]:
pipeline.fit(X, y)

predictions = pipeline.predict(X_test).astype(int).ravel()

submission = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Survived': predictions
})

print('Submission shape:', submission.shape)
print('\nPrediction distribution:')
print(submission['Survived'].value_counts().sort_index())

display(submission.head())


Submission shape: (418, 2)

Prediction distribution:
Survived
0    259
1    159
Name: count, dtype: int64


,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1


## 7. Validate Submission

In [7]:
assert submission.shape == (418, 2)
assert list(submission.columns) == ['PassengerId', 'Survived']
assert submission['PassengerId'].equals(test['PassengerId'])
assert submission['Survived'].isin([0, 1]).all()
assert submission['PassengerId'].is_unique

print('All submission checks passed.')


All submission checks passed.


## 8. Save Submission

In [8]:
output_path = '../submissions/submission_06.csv'
submission.to_csv(output_path, index=False)

print(f'Saved: {output_path}')


Saved: ../submissions/submission_06.csv
